<a href="https://colab.research.google.com/github/vikrant48/AI-ML-All-AI-related-python-codes-/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq sentence-transformers chromadb pymupdf

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq
client = Groq(api_key=groq_api_key)
print(client)

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
import chromadb

db = chromadb.PersistentClient(path="./resume_db")   # Saves to disk
collection = db.get_or_create_collection(
    name='resumes',
    metadata={'hnsw:space': 'cosine'}   # Use cosine similarity
)

In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(file_path):
    text = ""
    with fitz.open(file_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

In [ ]:
resume_text = extract_text_from_pdf("Resume.pdf")

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
name = extract_name(resume_text)
print(name)

In [ ]:
resumes_list = [
    {
        "id": "1",
        "name": name,
        "years": 3,
        "text": resume_text
    }
]

In [ ]:
for i, resume in enumerate(resumes_list):
    vector = model.encode(resume['text']).tolist()

    collection.add(
        ids=[resume['id']],
        embeddings=[vector],
        documents=[resume['text']],
        metadatas=[{
            'name': resume['name'],
            'years_exp': resume['years']
        }]
    )

print(f"Stored {collection.count()} resumes")

In [ ]:
data = collection.get()

print(data.keys())

for i in range(len(data['ids'])):
    print(f"\nResume {i+1}")
    print("ID:", data['ids'][i])
    print("Name:", data['metadatas'][i]['name'])
    print("Experience:", data['metadatas'][i]['years_exp'])
    print("Text (first 300 chars):")
    print(data['documents'][i][:300])
    print("-" * 60)

In [ ]:
query = """
Looking for a Java backend engineer with 3+ years experience,
Spring Boot, AWS, microservices, REST APIs, and database knowledge
"""

In [ ]:
query_vector = model.encode(query).tolist()
print(query_vector)

In [ ]:
results = collection.query(
    query_embeddings=[query_vector],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)
print(results)

In [ ]:
for i in range(len(results['ids'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    dist = results['distances'][0][i]

    score = 1 - dist  # cosine similarity → higher = better

    print(f"\nRank {i+1}")
    print(f"Score: {score:.3f}")
    print(f"Name: {meta['name']}")
    print(f"Experience: {meta['years_exp']} years")
    print(f"Preview: {doc[:200]}...")
    print("-" * 50)

In [ ]:
context_parts = []

for i in range(len(results['ids'][0])):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    dist = results['distances'][0][i]

    score = 1 - dist

    context_parts.append(
        f"""Candidate {i+1}
Name: {meta['name']}
Experience: {meta['years_exp']} years
Match Score: {score:.3f}

Resume:
{doc[:1000]}
"""
    )

context = "\n\n----------------------\n\n".join(context_parts)

print(context[:1500])

In [ ]:
groq_response = client.chat.completions.create(
  model="llama-3.1-8b-instant",
  temperature=0.3,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
            {
            "role": "user",
            "content": f"""
Job Query:
{query}

Candidates:
{context}

Instructions:
1. Rank top candidates
2. Show Name
3. Show Match Score (0-100)
4. Give short reason
"""
        }


  ]
)

print(groq_response.choices[0].message.content)